# Model Context Protocol (MCP): Solusi Latihan

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/dteti-workshop/workshop-agent-migas/blob/main/colab/solusi/hari4/09-mcp-solusi.ipynb)

Ini **solusi lengkap** semua latihan bab ini (4 latihan) di Google Colab. Semua bagian `TODO` sudah terisi, jadi notebook ini tinggal dijalankan (**Runtime > Run all**, atau `Shift+Enter` per cell). Keluaran dari satu kali eksekusi dan penjelasan tiap latihan sudah disertakan; jawaban model berubah tiap kali dijalankan, jadi hasil Anda bisa sedikit berbeda. Kerjakan dulu [notebook latihannya](https://colab.research.google.com/github/dteti-workshop/workshop-agent-migas/blob/main/colab/hari4/09-mcp-latihan.ipynb); buka solusi ini untuk mencocokkan hasil Anda atau saat macet.

1. Pilih **File > Save a copy in Drive** agar perubahan tersimpan di akun Anda sendiri.
2. Jalankan tiga cell **Persiapan Colab** berurutan (sekitar 2-3 menit pada sesi baru).
3. Kerjakan dari atas ke bawah dengan `Shift+Enter`.

Kalau runtime terputus, file di `/content` (termasuk `.env`) hilang. Jalankan ulang Persiapan, lalu lanjutkan dari cell yang sedang Anda kerjakan.

## Persiapan Colab
Jalankan tiga cell berikut berurutan sebelum cell lain di notebook ini.

In [ ]:
# Persiapan 1 - Menyalin repository dan memasang pustaka (sekitar 2-3 menit pada sesi baru)
import os
import subprocess
import sys

REPO_URL = "https://github.com/dteti-workshop/workshop-agent-migas.git"
REPO = "/content/workshop-agent-migas" if os.path.isdir("/content") else os.path.abspath("workshop-agent-migas")
if not os.path.isdir(REPO):
    subprocess.run(["git", "clone", "-q", "-b", "main", REPO_URL, REPO], check=True)
os.chdir(REPO)

pasang = subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "colab/requirements-colab.txt", "-e", "."],
                        capture_output=True, text=True)
if pasang.returncode != 0:
    print(pasang.stderr[-3000:])
    raise RuntimeError("Instalasi pustaka gagal. Jalankan ulang cell ini.")

# "-e ." memasang common dan migas_shared agar subproses (server MCP) dapat mengimpornya; kernel yang sedang berjalan
# belum memuat instalasi baru, jadi folder repo juga ditambahkan ke sys.path
if REPO not in sys.path:
    sys.path.insert(0, REPO)
os.environ["PYTHONPATH"] = REPO + os.pathsep + os.environ.get("PYTHONPATH", "")
print("Repository dan pustaka siap. Folder kerja:", os.getcwd())

In [ ]:
# Persiapan 2 - Mengisi ID peserta dan API key (dilewati bila .env sudah berisi key)
import re
from getpass import getpass
from pathlib import Path

from dotenv import load_dotenv

env = Path(REPO) / ".env"


def rahasia(nama: str):
    """Nilai dari Colab Secrets (ikon kunci di panel kiri), atau None bila tidak ada."""
    try:
        from google.colab import userdata
        return userdata.get(nama)
    except Exception:  # noqa: BLE001 - bukan di Colab, secret belum dibuat, atau akses notebook belum diizinkan
        return None


if not (env.exists() and re.search(r"(?m)^ONLINE_API_KEY=(?!isi-dengan)\S+", env.read_text(encoding="utf-8"))):
    peserta = (rahasia("PARTICIPANT_ID") or input("ID peserta (nomor meja dan nama singkat, contoh P07-andi): ")).strip()
    kunci = (rahasia("ONLINE_API_KEY") or getpass("API key workshop (ketikan tidak ditampilkan): ")).strip()
    if not kunci:
        raise ValueError("API key belum diisi. Jalankan ulang cell ini.")
    isi = (Path(REPO) / ".env.example").read_text(encoding="utf-8")
    for nama, nilai in (("PARTICIPANT_ID", peserta or "P00-nama"), ("ONLINE_API_KEY", kunci)):
        isi = re.sub(rf"(?m)^{nama}=.*$", lambda _, n=nama, v=nilai: f"{n}={v}", isi)
    env.write_text(isi, encoding="utf-8")
    del kunci

# Nilai dari .env dimuat ulang; modul yang sudah diimpor dibuang agar membaca nilai terbaru
load_dotenv(env, override=True)
for modul in [m for m in sys.modules if m.split(".")[0] in ("common", "migas_shared", "rag_pipeline")]:
    del sys.modules[modul]
print("File .env siap untuk peserta", os.environ.get("PARTICIPANT_ID"))

In [ ]:
# Persiapan 3 - Memeriksa koneksi ke model
from migas_shared.llm import chat

try:
    print("Jawaban model:", chat("Balas hanya dengan kata: SIAP").strip())
except Exception as error:  # noqa: BLE001
    raise RuntimeError("Model tidak menjawab. Periksa API key (jalankan ulang Persiapan 2) dan koneksi internet. "
                       f"Pesan asli: {error}") from error

## Solusi Latihan 1: Menghubungkan ke Server MCP Context7
Jalankan sel satu per satu dengan Shift+Enter

Skrip ini:

- Menghubungkan ke server publik Context7 (https://mcp.context7.com/mcp)
- Mendaftar semua tool yang ditawarkan server
- Membuat agen yang memakai tool itu
- Meminta dokumentasi sebuah library dan menampilkan jawabannya

Kode async dijalankan lewat `jalankan(...)` dari `migas_shared.tampil`, yang bekerja di skrip biasa maupun
di Jupyter/Interactive Window.

In [ ]:
# Langkah 1 - Menyiapkan klien MCP dan model
import asyncio
import os

from dotenv import load_dotenv
from langchain.agents import create_agent
from langchain_mcp_adapters.client import MultiServerMCPClient
from langchain_openai import ChatOpenAI

from migas_shared.tampil import jalankan

load_dotenv()

# Klien membuka sesi per panggilan dan menutupnya sendiri, jadi tidak ada yang perlu ditutup antar sel.
client = MultiServerMCPClient(
    {"context7": {"transport": "streamable_http", "url": "https://mcp.context7.com/mcp"}}
)

model = ChatOpenAI(
    model=os.getenv("ONLINE_MODEL"),
    base_url=os.getenv("ONLINE_BASE_URL"),
    api_key=os.getenv("ONLINE_API_KEY"),
    reasoning_effort=os.getenv("LLM_REASONING_EFFORT", "none"),
)

## Langkah 2 - Mendaftar tool dari Context7
Kalau koneksi gagal (misalnya tanpa internet), pesan error dicetak dan daftar tool dibiarkan kosong.

In [ ]:
try:
    tools = jalankan(asyncio.wait_for(client.get_tools(), timeout=30))
    print("Tool yang tersedia dari Context7:")
    for tool in tools:
        print(f"   - {tool.name}: {tool.description.strip().splitlines()[0][:100]}")
    print()
except Exception as error:
    tools = []
    print(f"Tidak dapat menyelesaikan: {type(error).__name__}: {error}")
    print("Periksa koneksi internet Anda atau coba lagi nanti.")

Tool yang tersedia dari Context7:
   - resolve-library-id: Resolves a package/product name to a Context7-compatible library ID and returns matching libraries.
   - query-docs: Retrieves and queries up-to-date documentation and code examples from Context7 for any programming l



## Langkah 3 - Membuat agen dan meminta dokumentasi
Agen memutuskan sendiri tool Context7 mana yang dipanggil untuk mengambil dokumentasi FastAPI terbaru.

In [ ]:
agent = create_agent(model, tools)

pertanyaan = (
    "Bagaimana cara membuat endpoint POST dengan FastAPI yang menerima JSON dan divalidasi dengan model Pydantic? "
    "Ambil dokumentasi terbaru. Jawab singkat dalam Bahasa Indonesia dengan satu contoh kode pendek."
)
print(f"Pengguna: {pertanyaan}\n")
try:
    response = jalankan(asyncio.wait_for(agent.ainvoke({"messages": [("human", pertanyaan)]}), timeout=180))
    panggilan = [c["name"] for m in response["messages"] for c in getattr(m, "tool_calls", [])]
    print(f"Tool MCP yang dipanggil: {panggilan}\n")
    print(f"Agen: {response['messages'][-1].content}")
except Exception as error:
    print(f"Tidak dapat menyelesaikan: {type(error).__name__}: {error}")
    print("Periksa koneksi internet Anda atau coba lagi nanti.")

Pengguna: Bagaimana cara membuat endpoint POST dengan FastAPI yang menerima JSON dan divalidasi dengan model Pydantic? Ambil dokumentasi terbaru. Jawab singkat dalam Bahasa Indonesia dengan satu contoh kode pendek.

Tool MCP yang dipanggil: ['resolve-library-id', 'query-docs']

Agen: ## Cara Membuat Endpoint POST dengan FastAPI + Validasi Pydantic

Deklarasikan model Pydantic dengan `BaseModel`, lalu jadikan sebagai parameter tipe pada endpoint POST. FastAPI otomatis membaca JSON body, memvalidasinya, dan menampilkan dokumentasi (Swagger).

```python
from fastapi import FastAPI
from pydantic import BaseModel

app = FastAPI()

class Item(BaseModel):
    name: str
    price: float
    tax: float | None = None   # optional

@app.post("/items/")
async def create_item(item: Item):
    return {"message": "Item dibuat", "data": item}
```

**Penjelasan singkat:**
- `Item` adalah model Pydantic yang mendefinisikan skema JSON yang diharapkan.
- Jika JSON yang dikirim tidak sesuai (misal `price` 

### Penjelasan Solusi Latihan 9.1: terhubung ke server MCP publik

MCP memisahkan tool dari agen: tool hidup di server, dan klien cukup menyambung lalu mengambil daftarnya.

- `TODO(B9.1)`: `MultiServerMCPClient` dengan satu server `context7` lewat transport `streamable_http` ke URL publik.
- `TODO(B9.2)`: `client.get_tools()` adalah fungsi async. `jalankan(...)` dari `migas_shared.tampil` menjalankannya di notebook, dan `asyncio.wait_for(..., timeout=30)` mencegah sel menggantung bila server tidak merespons.
- `TODO(B9.3)`: tool MCP langsung dipakai `create_agent`, sama seperti tool lokal.

Server menawarkan dua tool, `resolve-library-id` dan `query-docs`. Agen memanggil keduanya berurutan: yang pertama mencari ID pustaka FastAPI, yang kedua mengambil dokumentasinya, lalu ia menyusun jawaban dengan contoh kode. Latihan ini butuh internet dan server pihak ketiga; bila server sedang tidak tersedia, sel mencetak pesan error dan daftar tool dibiarkan kosong, bukan berhenti dengan traceback.

## Solusi Latihan 2: Agen Multi-Tool dengan MCP
Jalankan sel satu per satu dengan Shift+Enter

Menggabungkan tool dari server MCP (lapangan, stdio) dengan tool yang Anda tulis sendiri di Python,
dalam SATU agen. Agen memilih tool dari deskripsinya, tidak peduli dari mana tool itu berasal.

- Tool MCP: hitung_water_cut, konversi_satuan, cek_status_sumur (`field_stdio_server.py`)
- Tool lokal: hitung_kerugian_produksi (ditulis di file ini)

Kode async dijalankan lewat `jalankan(...)` dari `migas_shared.tampil`, yang bekerja di skrip biasa maupun
di Jupyter/Interactive Window.

In [ ]:
# Langkah 1 - Impor dan model
import os
import sys

from dotenv import load_dotenv
from langchain.agents import create_agent
from langchain_core.tools import tool
from langchain_mcp_adapters.client import MultiServerMCPClient
from langchain_openai import ChatOpenAI
from pydantic import BaseModel, Field

from migas_shared.config import ROOT
from migas_shared.tampil import jalankan

load_dotenv()

PATH_SERVER = ROOT / "hari4" / "09-mcp" / "code" / "servers" / "field_stdio_server.py"

model = ChatOpenAI(
    model=os.getenv("ONLINE_MODEL"),
    base_url=os.getenv("ONLINE_BASE_URL"),
    api_key=os.getenv("ONLINE_API_KEY"),
    reasoning_effort=os.getenv("LLM_REASONING_EFFORT", "none"),
)

## Langkah 2 - Tool lokal
Skema masukan Pydantic dan tool `hitung_kerugian_produksi`, ditulis langsung di Python.

In [ ]:
class KerugianInput(BaseModel):
    """Skema masukan tool kerugian produksi."""

    downtime_jam: float = Field(ge=0, description="Lama sumur tidak berproduksi, dalam jam")
    laju_normal_bopd: float = Field(ge=0, description="Laju produksi minyak normal sumur dalam bopd")


@tool(args_schema=KerugianInput)
def hitung_kerugian_produksi(downtime_jam: float, laju_normal_bopd: float) -> str:
    """Hitung perkiraan minyak yang hilang (dalam barel) akibat downtime sebuah sumur.
    Pakai tool ini untuk pertanyaan tentang kehilangan produksi dari downtime."""
    print(f"   [tool lokal] hitung_kerugian_produksi({downtime_jam}, {laju_normal_bopd})")
    hilang = laju_normal_bopd * downtime_jam / 24
    return f"Perkiraan kehilangan produksi: {hilang:.1f} barel minyak"

## Langkah 3 - Menggabungkan tool MCP dan tool lokal
Klien membuka sesi per panggilan dan menutupnya sendiri, jadi tidak ada yang perlu ditutup antar sel.

In [ ]:
client = MultiServerMCPClient(
    {"lapangan": {"transport": "stdio", "command": sys.executable, "args": [str(PATH_SERVER)]}}
)
tools_mcp = jalankan(client.get_tools())
semua_tool = [*tools_mcp, hitung_kerugian_produksi]  # gabungkan: MCP + lokal
print(f"Tool MCP:   {[t.name for t in tools_mcp]}")
print(f"Tool lokal: {[hitung_kerugian_produksi.name]}\n")

Tool MCP:   ['hitung_water_cut', 'konversi_satuan', 'cek_status_sumur']
Tool lokal: ['hitung_kerugian_produksi']



## Langkah 4 - Membuat agen
Satu agen menerima semua tool, tanpa peduli asalnya.

In [ ]:
agent = create_agent(
    model,
    semua_tool,
    system_prompt="Anda asisten teknis lapangan migas. Pakai tool, jangan menghitung sendiri. Jawab singkat dalam Bahasa Indonesia.",
)


def tanya_agen(pertanyaan: str) -> None:
    print(f"Pengguna: {pertanyaan}")
    response = jalankan(agent.ainvoke({"messages": [("human", pertanyaan)]}))
    panggilan = [c["name"] for m in response["messages"] for c in getattr(m, "tool_calls", [])]
    print(f"   Tool yang dipanggil: {panggilan}")
    print(f"Agen: {response['messages'][-1].content}\n")

## Langkah 5 - Pertanyaan 1: memakai tool MCP

In [ ]:
tanya_agen("Berapa water cut sumur dengan produksi 95 bopd minyak dan 380 bwpd air?")  # tool MCP

Pengguna: Berapa water cut sumur dengan produksi 95 bopd minyak dan 380 bwpd air?
   Tool yang dipanggil: ['hitung_water_cut']
Agen: Water cut sumur tersebut adalah **80%**, dengan total cairan produksi **475 bpd** (95 bopd minyak + 380 bwpd air).



## Langkah 6 - Pertanyaan 2: memakai tool lokal

In [ ]:
tanya_agen("Sumur SR-020 mati 18 jam dengan laju normal 120 bopd. Berapa barel minyak yang hilang?")  # tool lokal

Pengguna: Sumur SR-020 mati 18 jam dengan laju normal 120 bopd. Berapa barel minyak yang hilang?
   [tool lokal] hitung_kerugian_produksi(18.0, 120.0)
   Tool yang dipanggil: ['hitung_kerugian_produksi']
Agen: Kehilangan produksi sumur SR-020 adalah **90 barel minyak**.



## Langkah 7 - Pertanyaan 3: memakai keduanya

In [ ]:
tanya_agen("Cek status sumur SR-027, lalu hitung kehilangan produksinya bila ia mati 10 jam pada laju normalnya.")  # keduanya

Pengguna: Cek status sumur SR-027, lalu hitung kehilangan produksinya bila ia mati 10 jam pada laju normalnya.
   [tool lokal] hitung_kerugian_produksi(10.0, 95.0)
   Tool yang dipanggil: ['cek_status_sumur', 'hitung_kerugian_produksi']
Agen: **Ringkasan Status Sumur SR-027:**
- **Area:** Seruni
- **Pompa:** ESP
- **Status:** Produksi
- **Minyak:** 95 bopd
- **Air:** 380 bwpd
- **Catatan:** Pembacaan H₂S 12 ppm (22 Juli 2026) — dalam investigasi HSE.

**Kehilangan produksi jika mati 10 jam:**
- **~ 39,6 barel minyak.**



### Penjelasan Solusi Latihan 9.2: tool MCP dan tool lokal dalam satu agen

Agen tidak peduli dari mana sebuah tool berasal. Tiga tool lapangan datang dari server MCP stdio yang dijalankan sebagai subproses, satu lagi ditulis langsung di Python.

- `TODO(B9.4)` dan `TODO(B9.5)`: tool lokal `hitung_kerugian_produksi` dengan skema Pydantic. Barel yang hilang dihitung `laju * jam / 24`.
- `TODO(B9.6)`: server stdio didefinisikan dengan `command=sys.executable` dan jalur skrip server. `sys.executable` memastikan subproses memakai Python yang sama dengan kernel, termasuk di Colab.
- `TODO(B9.7)`: tool MCP dan tool lokal digabung dalam satu daftar `semua_tool`.
- `TODO(B9.8)`: satu agen menerima semuanya.

Pada keluaran, pertanyaan water cut dijawab tool MCP, pertanyaan kehilangan produksi dijawab tool lokal (120 bopd selama 18 jam menjadi 90 barel), dan pertanyaan ketiga memakai keduanya: status SR-027 dari MCP, lalu kerugian 10 jam dari tool lokal. Daftar `Tool yang dipanggil` membuktikan agen memilih berdasarkan deskripsi tool, bukan berdasarkan asalnya.

## Solusi Latihan 3 (Bonus): Server MCP Buatan Sendiri

Latihan ini berupa file Python, `hari4/09-mcp/latihan_3.py`. Di Colab file itu ditulis dari cell di bawah. Kode di bawah sudah lengkap. Jalankan cell untuk menyimpan file.

Baris pertama cell (`%%writefile ...`) menentukan nama file dan tidak boleh diubah.

In [ ]:
%%writefile hari4/09-mcp/latihan_3.py
"""
Server MCP buatan sendiri untuk Latihan 3 (Bonus): informasi work order.

Server stdio ini membuka dua tool hanya-baca:
- daftar_wo_terbuka(area)           : work order terbuka di sebuah area
- hitung_kerugian_downtime(...)     : perkiraan biaya downtime

Dipakai oleh latihan_4.py (klien yang menjalankannya sebagai subproses).
"""

import sys

from mcp.server.fastmcp import FastMCP

mcp = FastMCP("work-order")

WORK_ORDER = {
    "Seruni": [
        ("WO-26-12501", "Kebocoran seal ESP SR-020", "tinggi"),
        ("WO-26-12502", "Lampu penerangan manifold berkedip", "rendah"),
    ],
    "Kenanga": [("WO-26-12611", "Sucker rod KN-033 patah, menunggu rig", "tinggi")],
    "Melati": [],
}


@mcp.tool()
def daftar_wo_terbuka(area: str) -> str:
    """
    Daftar work order pemeliharaan yang masih terbuka di sebuah area.

    Args:
        area: Nama area: 'Seruni', 'Kenanga', atau 'Melati'

    Returns:
        Satu baris per work order: id, deskripsi, dan prioritas.
    """
    daftar = WORK_ORDER.get(area.strip().capitalize())
    if daftar is None:
        raise ValueError(f"Area '{area}' tidak dikenal. Area yang ada: {', '.join(WORK_ORDER)}")
    if not daftar:
        return f"Area {area}: tidak ada work order terbuka"
    return "\n".join(f"{wo} | {deskripsi} | prioritas {prioritas}" for wo, deskripsi, prioritas in daftar)


@mcp.tool()
def hitung_kerugian_downtime(downtime_jam: float, laju_bopd: float, harga_usd_per_bbl: float = 70.0) -> str:
    """
    Hitung perkiraan kerugian akibat downtime sebuah sumur.

    Args:
        downtime_jam: Lama downtime dalam jam
        laju_bopd: Laju produksi minyak normal dalam bopd
        harga_usd_per_bbl: Harga minyak dalam USD per barel (bawaan 70)

    Returns:
        Barel yang hilang dan perkiraan kerugian dalam USD.
    """
    if downtime_jam < 0 or laju_bopd < 0:
        raise ValueError("downtime_jam dan laju_bopd tidak boleh negatif")
    barel = laju_bopd * downtime_jam / 24
    return f"Kehilangan {barel:.1f} barel, perkiraan kerugian USD {barel * harga_usd_per_bbl:,.0f}"


if __name__ == "__main__":
    print("Server MCP work order (stdio) berjalan...", file=sys.stderr)
    mcp.run(transport="stdio")

Overwriting hari4/09-mcp/latihan_3.py


### Penjelasan Solusi Latihan 9.3: server MCP buatan sendiri

Server ini ditulis dengan `FastMCP` dan jalan lewat stdio. Notebook hanya menyimpannya sebagai file `hari4/09-mcp/latihan_3.py`; server baru benar-benar berjalan di latihan berikutnya, saat klien menjalankannya sebagai subproses.

- `TODO(B9.9)`: `daftar_wo_terbuka` dihias `@mcp.tool()`. Docstring-nya, termasuk bagian `Args` dan `Returns`, menjadi deskripsi yang dibaca model. Area yang tidak dikenal memicu `ValueError` dengan daftar area yang valid.
- `TODO(B9.10)`: `hitung_kerugian_downtime` menolak nilai negatif dan menghitung barel serta kerugian USD.

Raise `ValueError` dengan pesan yang jelas lebih baik daripada mengembalikan teks kosong: pesannya sampai ke klien, dan model bisa memperbaiki permintaannya. Keluaran sel ini hanya `Overwriting ...`, karena `%%writefile` tidak menjalankan apa pun.

## Solusi Latihan Bonus: Server MCP Buatan Sendiri + Klien Pengujinya
Jalankan sel satu per satu dengan Shift+Enter

Server ada di `latihan_3.py`. Skrip ini:

1. Menjalankan server sebagai subproses (stdio) dan mendaftar tool-nya
2. Menguji tool LANGSUNG dengan input yang valid dan tidak valid (tanpa model), diperiksa di kode
3. Menghubungkan agen ke server dan mengajukan satu pertanyaan gabungan

Kode async dijalankan lewat `jalankan(...)` dari `migas_shared.tampil`, yang bekerja di skrip biasa maupun
di Jupyter/Interactive Window.

In [ ]:
# Langkah 1 - Impor dan model
import os
import sys

from dotenv import load_dotenv
from langchain.agents import create_agent
from langchain_core.tools import ToolException
from langchain_mcp_adapters.client import MultiServerMCPClient
from langchain_openai import ChatOpenAI

from migas_shared.config import ROOT
from migas_shared.tampil import jalankan

load_dotenv()

PATH_SERVER = ROOT / "hari4" / "09-mcp" / "latihan_3.py"

model = ChatOpenAI(
    model=os.getenv("ONLINE_MODEL"),
    base_url=os.getenv("ONLINE_BASE_URL"),
    api_key=os.getenv("ONLINE_API_KEY"),
    reasoning_effort=os.getenv("LLM_REASONING_EFFORT", "none"),
)

## Langkah 2 - Mendaftar tool dari server buatan sendiri
Klien membuka sesi per panggilan dan menutupnya sendiri, jadi tidak ada yang perlu ditutup antar sel.

In [ ]:
client = MultiServerMCPClient(
    {"work_order": {"transport": "stdio", "command": sys.executable, "args": [str(PATH_SERVER)]}}
)
tools = jalankan(client.get_tools())
tool_per_nama = {t.name: t for t in tools}
print(f"Tool dari server buatan sendiri: {list(tool_per_nama)}\n")

Tool dari server buatan sendiri: ['daftar_wo_terbuka', 'hitung_kerugian_downtime']



## Langkah 3 - Uji langsung 1: daftar work order terbuka
Pengujian tanpa model, diperiksa di kode. Hasil `daftar_wo_terbuka('Kenanga')` harus memuat WO-26-12611.

In [ ]:
print("Uji langsung (tanpa model):")
lolos = 0

hasil_daftar = str(jalankan(tool_per_nama["daftar_wo_terbuka"].ainvoke({"area": "Kenanga"})))
ok = "WO-26-12611" in hasil_daftar
lolos += ok
print(f"   {'LOLOS' if ok else 'GAGAL'}  daftar_wo_terbuka('Kenanga') memuat WO-26-12611")

Uji langsung (tanpa model):
   LOLOS  daftar_wo_terbuka('Kenanga') memuat WO-26-12611


## Langkah 4 - Uji langsung 2: kerugian downtime
12 jam pada 100 bopd harus menghasilkan 50 barel dan USD 3.500.

In [ ]:
hasil_kerugian = str(jalankan(tool_per_nama["hitung_kerugian_downtime"].ainvoke({"downtime_jam": 12, "laju_bopd": 100})))
ok = "50.0 barel" in hasil_kerugian and "3,500" in hasil_kerugian
lolos += ok
print(f"   {'LOLOS' if ok else 'GAGAL'}  hitung_kerugian_downtime(12 jam, 100 bopd) = 50 barel, USD 3.500")

   LOLOS  hitung_kerugian_downtime(12 jam, 100 bopd) = 50 barel, USD 3.500


## Langkah 5 - Uji langsung 3: input tidak valid
Area yang tidak dikenal harus ditolak dengan error yang jelas.

In [ ]:
try:
    hasil_salah = str(jalankan(tool_per_nama["daftar_wo_terbuka"].ainvoke({"area": "Atlantis"})))
    ok = "tidak dikenal" in hasil_salah  # adaptor MCP mengembalikan error server sebagai teks hasil
except ToolException:
    ok = True  # versi adaptor lain melempar ToolException; keduanya berarti area ditolak
lolos += ok
print(f"   {'LOLOS' if ok else 'GAGAL'}  area tidak dikenal ditolak dengan error")

print(f"\nHasil uji langsung: {lolos}/3 lolos\n")

   LOLOS  area tidak dikenal ditolak dengan error

Hasil uji langsung: 3/3 lolos



## Langkah 6 - Agen memakai server
Satu pertanyaan gabungan: daftar work order terbuka dan kerugian downtime.

In [ ]:
agent = create_agent(
    model,
    tools,
    system_prompt="Anda asisten pemeliharaan. Pakai tool, jangan menebak. Jawab singkat dalam Bahasa Indonesia.",
)
pertanyaan = "Work order apa yang terbuka di Kenanga, dan berapa kerugian bila sumurnya mati 48 jam pada laju 90 bopd?"
print(f"Pengguna: {pertanyaan}")
response = jalankan(agent.ainvoke({"messages": [("human", pertanyaan)]}))
panggilan = [c["name"] for m in response["messages"] for c in getattr(m, "tool_calls", [])]
print(f"   Tool yang dipanggil: {panggilan}")
print(f"Agen: {response['messages'][-1].content}")

Pengguna: Work order apa yang terbuka di Kenanga, dan berapa kerugian bila sumurnya mati 48 jam pada laju 90 bopd?
   Tool yang dipanggil: ['daftar_wo_terbuka', 'hitung_kerugian_downtime']
Agen: Berikut hasilnya:

**Work Order Terbuka di Area Kenanga:**
- **WO-26-12611** | Sucker rod KN-033 patah, menunggu rig | Prioritas: Tinggi

**Perkiraan Kerugian Downtime (48 jam, laju 90 bopd):**
- Barel yang hilang: **180 barel**
- Perkiraan kerugian: **USD 12.600** (dengan asumsi harga $70/bbl)


### Penjelasan Solusi Latihan 9.4: klien penguji untuk server buatan sendiri

Sebelum diberikan ke agen, tool server diuji langsung tanpa model, dengan tiga pemeriksaan di kode.

- `TODO(B9.11)` dan `TODO(B9.12)`: klien stdio untuk server `work_order`, lalu daftar tool diubah menjadi kamus nama ke tool.
- `TODO(B9.13)`: `daftar_wo_terbuka('Kenanga')` harus memuat WO-26-12611.
- `TODO(B9.14)`: area `Atlantis` harus ditolak. Di sini ada pelajaran: adaptor `langchain-mcp-adapters` versi yang dipakai kursus (0.3.x) tidak melempar `ToolException` bila server mengalami error; ia mengembalikan pesan error sebagai teks hasil (`Error executing tool ...`). Karena itu pemeriksaan membaca teksnya ("tidak dikenal"), dan `except ToolException` dipertahankan untuk adaptor yang melempar.
- `TODO(B9.15)`: agen dibuat dari tool server dengan prompt pemeliharaan.

Ketiga uji langsung lolos pada keluaran. Pertanyaan gabungan di akhir memanggil kedua tool: work order terbuka di Kenanga, dan kerugian 48 jam pada 90 bopd (180 barel, sekitar USD 12.600 dengan harga USD 70 per barel). Menguji tool lebih dulu tanpa model memisahkan dua sumber kegagalan: bug di tool dan kebingungan model.